# Chapter 7: NumPy and pandas
**Companion notebook** · Book 1: Python and Data Foundations for ML

Every code example from Chapter 7, in the same order as the book. Part 1 uses NumPy on small lists; from Part 2 onwards we work with all 45,211 bank customers (needs an internet connection).

## 1. NumPy
### Creating arrays

In [ ]:
import numpy as np

np.set_printoptions(suppress=True)
balances = np.array([2143, 29, 2, 1506, 1])
print(balances)
print(type(balances))
print(balances.dtype)
print(balances.shape)

### Vectorised maths: no loop needed

In [ ]:
print(balances + 500)
print((balances * 1.07).round(2))
print(balances / 1000)

deposits = np.array([100, 200, 300, 400, 500])
print(balances + deposits)

### Statistics in one line

In [ ]:
print(balances.sum(), balances.mean(), balances.max(), balances.min())
print(np.median(balances))
print(balances.std().round(1))

### Indexing, slicing and boolean masks

In [ ]:
print(balances[0], balances[-1])
print(balances[1:4])

is_low = balances < 100
print(is_low)
print(balances[is_low])
print(balances[balances > 1000])
print(is_low.sum())

### np.where: choose a value for every item

In [ ]:
labels = np.where(balances >= 1000, "healthy", "low")
print(labels)

### Two-dimensional arrays

In [ ]:
customers = np.array([
    [58, 2143],
    [44, 29],
    [33, 2],
])
print(customers.shape)
print(customers[0])
print(customers[:, 1])
print(customers.mean(axis=0).round(1))

### Why NumPy is fast (your times will differ)

In [ ]:
import time

numbers = list(range(1_000_000))
array = np.arange(1_000_000)

start = time.perf_counter()
doubled_list = [n * 2 for n in numbers]
list_seconds = time.perf_counter() - start

start = time.perf_counter()
doubled_array = array * 2
array_seconds = time.perf_counter() - start

print(f"List: {list_seconds:.4f} s, NumPy: {array_seconds:.4f} s")
print(f"NumPy was about {list_seconds / array_seconds:.0f} times faster")

## 2. pandas Series

In [ ]:
import pandas as pd

ages = pd.Series([58, 44, 33, 47, 33], name="age")
print(ages)
print(ages.mean())
print(ages.value_counts())

## 3. Loading the bank data into a DataFrame (needs an internet connection)

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd

bank_data = fetch_ucirepo(id=222)
bank = bank_data.data.features.copy()
bank["subscribed"] = bank_data.data.targets["y"]
bank = bank.fillna("unknown")
print(bank.shape)

### A quick look

In [ ]:
print(bank.columns.tolist())
print(bank[["age", "job", "balance", "subscribed"]].head())

## 4. Selecting data
### Columns

In [ ]:
ages = bank["age"]
print(type(ages))

small = bank[["age", "job", "balance"]]
print(type(small))
print(small.shape)

### Rows with iloc (by position) and loc (by label)

In [ ]:
print(bank.iloc[0])

In [ ]:
print(bank.iloc[0:3, 0:3])
print(bank.loc[0:2, ["age", "job"]])
print(bank.loc[2, "balance"])

## 5. Filtering rows

In [ ]:
rich = bank[bank["balance"] > 50000]
print(len(rich))
print(rich[["age", "job", "balance"]])

In [ ]:
young_students = bank[(bank["job"] == "student") & (bank["age"] < 25)]
print(len(young_students))

loan_or_default = bank[(bank["loan"] == "yes") | (bank["default"] == "yes")]
print(len(loan_or_default))

no_housing_loan = bank[~(bank["housing"] == "yes")]
print(len(no_housing_loan))

In [ ]:
office_jobs = bank[bank["job"].isin(["admin.", "management"])]
print(len(office_jobs))

thirties = bank[bank["age"].between(30, 39)]
print(len(thirties))

print(len(bank.query("balance < 0 and loan == 'yes'")))

## 6. Sorting, counting and new columns

In [ ]:
top = bank.sort_values("balance", ascending=False)
print(top[["age", "job", "balance"]].head(3))

print(bank["marital"].value_counts())
print(bank["subscribed"].value_counts(normalize=True).round(3))

In [ ]:
bank["balance_k"] = bank["balance"] / 1000
bank["subscribed_num"] = bank["subscribed"].map({"yes": 1, "no": 0})
bank["is_overdrawn"] = bank["balance"] < 0
print(bank[["balance", "balance_k", "subscribed", "subscribed_num", "is_overdrawn"]].head(3))
print(bank["is_overdrawn"].sum())

In [ ]:
import numpy as np

bank["balance_band"] = np.where(bank["balance"] >= 1000, "healthy", "low")
print(bank["balance_band"].value_counts())

def life_stage(age):
    if age < 30:
        return "young"
    if age < 60:
        return "working"
    return "senior"

bank["life_stage"] = bank["age"].apply(life_stage)
print(bank["life_stage"].value_counts())

## 7. Grouping and summarising

In [ ]:
print(bank.groupby("marital")["balance"].mean().round())

In [ ]:
job_rates = bank.groupby("job")["subscribed_num"].mean().sort_values(ascending=False)
print(job_rates.head(3).round(3))

In [ ]:
summary = bank.groupby("life_stage").agg(
    customers=("age", "size"),
    average_age=("age", "mean"),
    median_balance=("balance", "median"),
    subscription_rate=("subscribed_num", "mean"),
)
print(summary.round(2))

In [ ]:
print(pd.crosstab(bank["housing"], bank["subscribed"], normalize="index").round(3))

In [ ]:
table = bank.pivot_table(index="life_stage", columns="loan",
                         values="subscribed_num", aggfunc="mean")
print(table.round(3))

## 8. Combining tables
### merge

In [ ]:
job_info = pd.DataFrame({
    "job": ["admin.", "management", "student", "retired"],
    "sector": ["office", "office", "education", "none"],
})
merged = bank.merge(job_info, on="job", how="left")
print(merged[["job", "sector"]].head())
print(merged["sector"].isna().sum())

### concat

In [ ]:
may_calls = bank[bank["month"] == "may"].head(2)
june_calls = bank[bank["month"] == "jun"].head(2)
both = pd.concat([may_calls, june_calls])
print(both[["age", "job", "month"]])

## 9. Saving your results

In [ ]:
summary.to_csv("life_stage_summary.csv")
check = pd.read_csv("life_stage_summary.csv")
print(check.round(2))

## 10. Mini project: a customer segment report

In [ ]:
bank["age_group"] = pd.cut(bank["age"], bins=[17, 29, 39, 49, 59, 100],
                           labels=["18-29", "30-39", "40-49", "50-59", "60+"])

report = bank.groupby("age_group", observed=True).agg(
    customers=("age", "size"),
    median_balance=("balance", "median"),
    overdrawn_share=("is_overdrawn", "mean"),
    subscription_rate=("subscribed_num", "mean"),
)
print(report.round(3))

share = report["customers"] / report["customers"].sum()
print(share.round(3))

In [ ]:
best = report["subscription_rate"].idxmax()
print(f"Best segment: {best}, rate {report.loc[best, 'subscription_rate']:.1%}")
print(f"Overall rate: {bank['subscribed_num'].mean():.1%}")

In [ ]:
import time

customers_list = bank.to_dict("records")

start = time.perf_counter()
totals, yeses = {}, {}
for c in customers_list:
    totals[c["job"]] = totals.get(c["job"], 0) + 1
    yeses[c["job"]] = yeses.get(c["job"], 0) + c["subscribed_num"]
loop_rates = {job: yeses[job] / totals[job] for job in totals}
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
pandas_rates = bank.groupby("job")["subscribed_num"].mean()
pandas_seconds = time.perf_counter() - start

print(round(loop_rates["student"], 3), round(pandas_rates["student"], 3))
print(f"Loop: {loop_seconds:.4f} s, pandas: {pandas_seconds:.4f} s")

## Try it yourself
Use the empty cells below for the exercises at the end of Chapter 7.

---
**Dataset credit:** S. Moro, P. Rita and P. Cortez, *Bank Marketing* dataset, UCI Machine Learning Repository (2014), https://doi.org/10.24432/C5K306. Licensed under CC BY 4.0.